# IEEE-30 Wildfire-Aware Dispatch Optimization Validation

This notebook demonstrates the current `experiments/test` wildfire-aware predict-then-optimize workflow for GridFM using the corrected single-scenario extraction path.

## Decision Variables

The optimization keeps the current extended decision vector:

$$u = [u_{Pg}; u_{\delta}]$$

where `u_Pg` is active generation at controllable PV buses and `u_delta` is load shedding at PQ buses in MW.

## Current Objective

$$J(u)=\lambda_{gen} \sum (u_{Pg}-P_{g,base})^2 + \lambda_{shed} \sum u_{\delta} + \lambda_{wf} \cdot wildfire\_penalty(u)$$

The wildfire term is computed from branch loading using a soft activation threshold:

$$wildfire\_penalty = \sum_{\ell} w_\ell \max(0, loading_\ell - \eta_\ell)^2$$

with default `w_l = 1.0` and `eta_l = 0.7`.

## Presentation Notes

- the code now extracts one 30-bus graph from the batched IEEE-30 test loader
- the exact decision dimension is determined at runtime from that extracted graph
- the notebook should be presented as a wildfire-aware surrogate optimization prototype, not as a final AC-OPF-quality result


## Setup

In [47]:
import sys
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve paths robustly whether the notebook is launched from repo root or experiments/test
cwd = Path.cwd().resolve()
if (cwd / "tests").exists() and (cwd / "experiments").exists():
    repo_root = cwd
    notebook_dir = repo_root / "experiments" / "test"
else:
    notebook_dir = cwd
    repo_root = notebook_dir.parent.parent

sys.path.insert(0, str(repo_root))

print(f"Notebook dir: {notebook_dir}")
print(f"Repo root: {repo_root}")
print(f"Python path: {sys.path[0]}")

from experiments.test import (
    ScenarioData,
    extract_scenario_from_batch,
    PVDispatchDecisionSpec,
    NeuralSolverWrapper,
    WildfirePenaltyEvaluator,
    DispatchOptimizationProblem,
    PipelineValidationHarness,
)

from gridfm_graphkit.io.param_handler import NestedNamespace, load_model
from gridfm_graphkit.datasets.powergrid_datamodule import LitGridDataModule

import yaml

print("[OK] Imports successful")


Notebook dir: C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit\experiments\test
Repo root: C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit
Python path: C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit


ImportError: cannot import name 'WildfirePenaltyEvaluator' from 'experiments.test' (C:\Users\Caleb Lu\OneDrive\Documents\GT\Extracurriculars\Research\Grid FM\Experiments\GridFM-graphkit\experiments\test\__init__.py)

## 1. Load Test Data (IEEE-30)

In [ ]:
# Load configuration
config_path = repo_root / "tests" / "config" / "gridFMv0.1_dummy.yaml"

with open(config_path, 'r', encoding='utf-8') as f:
    config_dict = yaml.safe_load(f)

args = NestedNamespace(**config_dict)

# Load datamodule - use test data directory
data_dir = repo_root / "tests" / "data"
datamodule = LitGridDataModule(args, data_dir=str(data_dir))
datamodule.setup(stage="test")

# Get one batched test sample
test_loader = datamodule.test_dataloader()[0]
batch = next(iter(test_loader))
ngraphs = int(batch.ptr.numel() - 1) if hasattr(batch, 'ptr') else 1

print("[OK] Loaded test batch")
print(f"  Num graphs in batch: {ngraphs}")
print(f"  Num nodes in batch: {batch.num_nodes}")
print(f"  Num edges in batch: {batch.num_edges}")
print(f"  PE shape: {batch.pe.shape}")


## 2. Extract Canonical Scenario

In [ ]:
# Extract a single scenario from the batch
node_normalizer = datamodule.node_normalizers[0]
edge_normalizer = datamodule.edge_normalizers[0]

scenario = extract_scenario_from_batch(
    batch,
    node_normalizer,
    edge_normalizer,
    scenario_idx=0,
    scenario_id="IEEE-30-test",
)

print(f"[OK] Extracted scenario: {scenario.scenario_id}")
print(f"  Num buses: {scenario.num_buses}")
print(f"  PQ buses: {int(np.sum(scenario.PQ_mask))}")
print(f"  PV buses: {int(np.sum(scenario.PV_mask))}")
print(f"  REF buses: {int(np.sum(scenario.REF_mask))}")
print(f"  PE dimension: {scenario.pe.shape[1]}")


## 3. Decision Variable Specification

In [ ]:
# Define decision variables
decision_spec = PVDispatchDecisionSpec(scenario)

print(f"✓ Decision specification created")
print(f"  Number of PV buses: {decision_spec.n_pv}")
print(f"  PV bus indices: {decision_spec.pv_bus_indices}")
print(f"  Baseline Pg at PV buses: {decision_spec.u_base}")
print(f"  Min bounds: {decision_spec.u_min}")
print(f"  Max bounds: {decision_spec.u_max}")

summary = decision_spec.get_summary()
print(f"\nDecision spec summary:")
for key, val in summary.items():
    print(f"  {key}: {val:.4f}")

## 4. Load Neural Solver Models

In [ ]:
# Load pretrained models
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# GNN model (v0.1)
model_gnn = load_model(args)
model_gnn_path = repo_root / "examples" / "models" / "GridFM_v0_1.pth"

if model_gnn_path.exists():
    checkpoint = torch.load(model_gnn_path, map_location=device)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        model_gnn.load_state_dict(checkpoint['state_dict'], strict=False)
    else:
        model_gnn.load_state_dict(checkpoint, strict=False)
    print(f"✓ Loaded GNN model from {model_gnn_path}")
else:
    print(f"⚠ GNN model not found at {model_gnn_path}")

# Create neural solver wrapper for GNN
solver_gnn = NeuralSolverWrapper(
    model_gnn,
    model_type="gnn",
    scenario=scenario,
    decision_spec=decision_spec,
    device=device,
)

print(f"✓ Created GNN solver wrapper")

## 5. Create Wildfire Penalty Evaluator


In [ ]:
# Create wildfire evaluator
wildfire_eval = WildfirePenaltyEvaluator(scenario, sn_mva=100.0)

print("[OK] Created wildfire evaluator")

baseline_wildfire = wildfire_eval.evaluate_baseline()
print("\nBaseline wildfire metrics:")
for key, val in baseline_wildfire.items():
    print(f"  {key}: {val}")


## 6. Full Pipeline Validation

In [ ]:
# Run full validation
validation_report = PipelineValidationHarness.full_validation(
    scenario,
    decision_spec,
    solver_gnn,
    solver_gps=None,
    overload_eval=wildfire_eval,
)

validation_all_passed = validation_report.get("all_passed", False)
PipelineValidationHarness.print_validation_report(dict(validation_report), verbose=True)
print(f"\nValidation all_passed: {validation_all_passed}")
print("Note: a False result here is currently expected if surrogate voltage predictions are not physically reasonable.")


## 7. Test Solver Predictions

In [ ]:
# Test prediction on baseline
u_base = decision_spec.u_base
pred = solver_gnn.predict_state(u_base)

print("Baseline prediction (first 5 buses):")
print(f"  Pd: {pred['Pd'][:5]}")
print(f"  Qd: {pred['Qd'][:5]}")
print(f"  Pg: {pred['Pg'][:5]}")
print(f"  Qg: {pred['Qg'][:5]}")
print(f"  Vm: {pred['Vm'][:5]}")
print(f"  Va: {pred['Va'][:5]}")

# Compute error vs baseline
baseline_state = scenario.get_baseline_state()
print(f"\nPrediction errors (RMSE):")
for key in ['Pd', 'Qd', 'Pg', 'Qg', 'Vm', 'Va']:
    rmse = np.sqrt(np.mean((pred[key] - baseline_state[key]) ** 2))
    print(f"  {key}: {rmse:.4f}")

## 8. Create Wildfire-Aware Optimization Problem


In [ ]:
# Create optimization problem
problem = DispatchOptimizationProblem(
    scenario=scenario,
    decision_spec=decision_spec,
    solver=solver_gnn,
    wildfire_eval=wildfire_eval,
    lambda_gen=1.0,
    lambda_shed=50.0,
    lambda_wf=10.0,
)

print("[OK] Created optimization problem")
print(f"  lambda_gen: {problem.lambda_gen}")
print(f"  lambda_shed: {problem.lambda_shed}")
print(f"  lambda_wf: {problem.lambda_wf}")


## 9. Evaluate Objective at Baseline


In [ ]:
# Evaluate objective at baseline
u_base = decision_spec.u_base
obj_base, details_base = problem.objective(u_base, return_details=True)

print("Objective at baseline dispatch:")
print(f"  Objective value: {obj_base:.4f}")
print(f"  Generator deviation cost: {details_base['generator_deviation_cost']:.4f}")
print(f"  Load shedding cost: {details_base['load_shedding_cost']:.4f}")
print(f"  Wildfire cost: {details_base['wildfire_cost']:.4f}")
print(f"  Active wildfire branches: {details_base['n_active_risk_branches']}")
print(f"  Max line loading: {details_base['max_loading']:.4f}")


## 10. Run Optimization

In [ ]:
# Run optimization (with reduced iterations for demo)
print("Starting optimization...")
opt_result = problem.optimize(
    method="L-BFGS-B",
    maxiter=50,
    verbose=True,
)

print(f"\nOptimization result:")
print(f"  Success: {opt_result['success']}")
print(f"  Message: {opt_result['message']}")
print(f"  Iterations: {opt_result['n_iter']}")
print(f"  Final objective: {opt_result['obj_opt']:.4f}")
print(f"  Final cost: {opt_result['cost_opt']:.4f}")
print(f"  Final wildfire cost: {opt_result['wildfire_opt']:.4f}")

## 11. Compare Baseline vs. Optimized

In [ ]:
# Compare baseline vs optimized
u_opt = opt_result['u_opt']
comparison = problem.compare_baseline_vs_optimized(u_opt)

print("\n" + "="*70)
print("BASELINE vs OPTIMIZED COMPARISON")
print("="*70)

print(f"\nBASELINE:")
for key, val in comparison['baseline'].items():
    if key != 'u':
        print(f"  {key}: {val:.4f}")

print(f"\nOPTIMIZED:")
for key, val in comparison['optimized'].items():
    if key != 'u':
        print(f"  {key}: {val:.4f}")

print(f"\nIMPROVEMENT:")
for key, val in comparison['improvement'].items():
    print(f"  {key}: {val:.4f}")

print("\n" + "="*70)

## 12. Visualize Optimization Progress

In [ ]:
# Plot optimization history
history = opt_result['history']

if len(history['objective']) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    axes[0, 0].plot(history['objective'], 'b-', linewidth=2)
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Objective Value')
    axes[0, 0].set_title('Total Objective')
    axes[0, 0].grid(True)
    
    axes[0, 1].plot(history['generator_deviation'], 'g-', linewidth=2)
    axes[0, 1].set_xlabel('Iteration')
    axes[0, 1].set_ylabel('Generator Deviation Cost')
    axes[0, 1].set_title('Generator Cost')
    axes[0, 1].grid(True)
    
    axes[1, 0].plot(history['wildfire'], 'r-', linewidth=2)
    axes[1, 0].set_xlabel('Iteration')
    axes[1, 0].set_ylabel('Wildfire Cost')
    axes[1, 0].set_title('Wildfire Component')
    axes[1, 0].grid(True)
    
    u_values = np.array(history['u'])
    for i in range(min(3, u_values.shape[1])):
        axes[1, 1].plot(u_values[:, i], label=f'Decision {i}')
    axes[1, 1].set_xlabel('Iteration')
    axes[1, 1].set_ylabel('Decision Value')
    axes[1, 1].set_title('Decision Trajectory')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('optimization_progress.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('[OK] Optimization progress plot saved as optimization_progress.png')
else:
    print('No optimization history to plot')


## 13. Summary and Next Steps

## Phase 2: Load Shedding with Reduced Penalties

Now extending to load shedding with lower penalty weights to encourage iteration.

In [ ]:
# Load GPS model (v0.2) - if available
model_gps_path = repo_root / "examples" / "models" / "GridFM_v0_2.pth"
has_gps = False

if model_gps_path.exists():
    config_dict_gps = config_dict.copy()
    config_dict_gps['model']['type'] = 'GPSTransformer'
    args_gps = NestedNamespace(**config_dict_gps)
    model_gps = load_model(args_gps)

    checkpoint_gps = torch.load(model_gps_path, map_location=device)
    if isinstance(checkpoint_gps, dict) and 'state_dict' in checkpoint_gps:
        model_gps.load_state_dict(checkpoint_gps['state_dict'], strict=False)
    else:
        model_gps.load_state_dict(checkpoint_gps, strict=False)

    model_gps = model_gps.to(device)
    model_gps.eval()
    has_gps = True
    print(f"[OK] Loaded GPS model from {model_gps_path}")
else:
    print(f"[WARN] GPS model not found at {model_gps_path}")


In [ ]:
# Create extended dispatch spec with load shedding
from experiments.test import ExtendedDispatchSpec

decision_spec_extended = ExtendedDispatchSpec(scenario, max_shed_fraction=1.0)
summary = decision_spec_extended.get_summary()

print("[OK] Extended decision spec created:")
print(f"  - PV buses (Pg): {decision_spec_extended.n_pv}")
print(f"  - PQ buses (shedding): {decision_spec_extended.n_pq}")
print(f"  - Total decision dimension: {decision_spec_extended.n_total}")
print(f"  - Max total shedding: {summary['shed_max_shed_total_MW']:.1f} MW")
print(f"  - Max shed as % of PQ load: {summary['shed_max_shed_pct_of_pq']:.1f}%")


In [ ]:
# Create solvers with extended dispatch spec
solver_gnn_extended = NeuralSolverWrapper(model_gnn, 'gnn', scenario, decision_spec_extended, device=device)
print("[OK] Created GNN solver with extended dispatch spec")

if has_gps:
    solver_gps = NeuralSolverWrapper(model_gps, 'gps', scenario, decision_spec_extended, device=device)
    print("[OK] Created GPS solver with extended dispatch spec")


In [ ]:
# GNN optimization with reduced penalties
print("\n" + "=" * 80)
print("GNN_v0.1 WITH LOAD SHEDDING (Reduced Penalties)")
print("=" * 80)
print("Penalty weights: lambda_gen=1.0, lambda_wf=10.0, lambda=0.01")

problem_gnn_shedding = DispatchOptimizationProblem(
    scenario, decision_spec_extended, solver_gnn_extended, wildfire_eval,
    lambda_gen=1.0,
    lambda_shed=50.0,
    lambda_wf=10.0,
)

result_gnn_shedding = problem_gnn_shedding.optimize(method='L-BFGS-B', maxiter=200, verbose=True)

print("\nOptimization completed:")
print(f"  Success: {result_gnn_shedding['success']}")
print(f"  Iterations: {result_gnn_shedding['n_iter']}")
print(f"  Message: {result_gnn_shedding['message']}")
print("\nObjective breakdown:")
print(f"  Cost deviation: {result_gnn_shedding['cost_opt']:.4f}")
print(f"  Shedding cost: {result_gnn_shedding['details'].get('cost_shedding', 0):.4f}")
print(f"  Wildfire cost: {result_gnn_shedding['wildfire_opt']:.4f}")
print(f"  TOTAL: {result_gnn_shedding['obj_opt']:.4f}")

u_opt_gnn_shed = result_gnn_shedding['u_opt']
Pg_change_gnn = np.sum(np.abs(u_opt_gnn_shed[:decision_spec_extended.n_pv] - decision_spec_extended.pv_spec.u_base))
shed_gnn = np.sum(u_opt_gnn_shed[decision_spec_extended.n_pv:])

print("\nDispatch changes:")
print(f"  Pg adjustment: {Pg_change_gnn:.4f} MW")
print(f"  Load shed: {shed_gnn:.4f} MW ({100 * decision_spec_extended.shed_spec.get_shed_fraction(u_opt_gnn_shed[decision_spec_extended.n_pv:]):.2f}% of PQ load)")


In [ ]:
# GPS optimization with reduced penalties (if available)
if has_gps:
    print("\n" + "=" * 80)
    print("GPS_v0.2 WITH LOAD SHEDDING (Reduced Penalties)")
    print("=" * 80)
    print("Penalty weights: lambda_gen=1.0, lambda_wf=10.0, lambda=0.01")

    problem_gps_shedding = DispatchOptimizationProblem(
        scenario, decision_spec_extended, solver_gps, wildfire_eval,
        lambda_gen=1.0,
        lambda_shed=50.0,
        lambda_wf=10.0,
    )

    result_gps_shedding = problem_gps_shedding.optimize(method='L-BFGS-B', maxiter=200, verbose=True)

    print("\nOptimization completed:")
    print(f"  Success: {result_gps_shedding['success']}")
    print(f"  Iterations: {result_gps_shedding['n_iter']}")
    print(f"  Message: {result_gps_shedding['message']}")
    print("\nObjective breakdown:")
    print(f"  Cost deviation: {result_gps_shedding['cost_opt']:.4f}")
    print(f"  Shedding cost: {result_gps_shedding['details'].get('cost_shedding', 0):.4f}")
    print(f"  Wildfire cost: {result_gps_shedding['wildfire_opt']:.4f}")
    print(f"  TOTAL: {result_gps_shedding['obj_opt']:.4f}")

    u_opt_gps_shed = result_gps_shedding['u_opt']
    Pg_change_gps = np.sum(np.abs(u_opt_gps_shed[:decision_spec_extended.n_pv] - decision_spec_extended.pv_spec.u_base))
    shed_gps = np.sum(u_opt_gps_shed[decision_spec_extended.n_pv:])

    print("\nDispatch changes:")
    print(f"  Pg adjustment: {Pg_change_gps:.4f} MW")
    print(f"  Load shed: {shed_gps:.4f} MW ({100 * decision_spec_extended.shed_spec.get_shed_fraction(u_opt_gps_shed[decision_spec_extended.n_pv:]):.2f}% of PQ load)")


In [ ]:
# Comparison Table: GNN vs GPS with Load Shedding
import pandas as pd

print("\n" + "=" * 90)
print("COMPARISON: GNN v0.1 vs GPS v0.2 with Wildfire Objective")
print("=" * 90)

comparison_data = {
    'Metric': [
        'Objective (Total)',
        'Cost Deviation',
        'Shedding Cost',
        'Wildfire Cost',
        'Iterations',
        'Pg Adjustment (MW)',
        'Load Shed (MW)',
        'Shed Fraction (%)'
    ],
    'GNN v0.1': [
        f"{result_gnn_shedding['obj_opt']:.2f}",
        f"{result_gnn_shedding['cost_opt']:.2f}",
        f"{result_gnn_shedding['details'].get('cost_shedding', 0):.2f}",
        f"{result_gnn_shedding['wildfire_opt']:.2f}",
        f"{result_gnn_shedding['n_iter']}",
        f"{Pg_change_gnn:.4f}",
        f"{shed_gnn:.4f}",
        f"{100 * decision_spec_extended.shed_spec.get_shed_fraction(u_opt_gnn_shed[decision_spec_extended.n_pv:]):.2f}%"
    ]
}

if has_gps:
    comparison_data['GPS v0.2'] = [
        f"{result_gps_shedding['obj_opt']:.2f}",
        f"{result_gps_shedding['cost_opt']:.2f}",
        f"{result_gps_shedding['details'].get('cost_shedding', 0):.2f}",
        f"{result_gps_shedding['wildfire_opt']:.2f}",
        f"{result_gps_shedding['n_iter']}",
        f"{Pg_change_gps:.4f}",
        f"{shed_gps:.4f}",
        f"{100 * decision_spec_extended.shed_spec.get_shed_fraction(u_opt_gps_shed[decision_spec_extended.n_pv:]):.2f}%"
    ]

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

if has_gps:
    print("\nInterpretation:")
    if result_gps_shedding['obj_opt'] < result_gnn_shedding['obj_opt']:
        print("  GPS produces the lower objective on this notebook run.")
    elif result_gps_shedding['obj_opt'] > result_gnn_shedding['obj_opt']:
        print("  GNN produces the lower objective on this notebook run.")
    else:
        print("  GNN and GPS produce matching objectives on this notebook run.")


In [ ]:
# Shedding Breakdown Analysis
print("\n" + "="*80)
print("LOAD SHEDDING BREAKDOWN BY BUS")
print("="*80)

# Extract PQ bus shed amounts for both models
shed_vector_gnn = u_opt_gnn_shed[decision_spec_extended.n_pv:]
shed_vector_gps = u_opt_gps_shed[decision_spec_extended.n_pv:] if has_gps else None

# Get PQ bus indices
pq_bus_indices = np.where(scenario.PQ_mask > 0)[0]
print(f"\nTotal PQ buses: {len(pq_bus_indices)}")

# Top 10 buses by shedding (GNN)
shed_by_bus_gnn = [(bus_idx, scenario.Pd_base[bus_idx], shed_vector_gnn[i],
                     100*shed_vector_gnn[i]/scenario.Pd_base[bus_idx] if scenario.Pd_base[bus_idx] > 0 else 0)
                    for i, bus_idx in enumerate(pq_bus_indices) if shed_vector_gnn[i] > 1e-4]

if shed_by_bus_gnn:
    shed_by_bus_gnn.sort(key=lambda x: x[2], reverse=True)
    print(f"\nGNN: Top buses by shedding amount:")
    print(f"{'Bus Idx':>8} {'Base Pd (MW)':>14} {'Shed (MW)':>12} {'% Shed':>10}")
    print("-" * 50)
    for bus_idx, base_pd, shed, pct in shed_by_bus_gnn[:10]:
        print(f"{bus_idx:>8} {base_pd:>14.4f} {shed:>12.4f} {pct:>10.2f}%")
else:
    print("GNN: No significant shedding (all < 0.0001 MW)")

# Top 10 buses by shedding (GPS)
if has_gps and shed_vector_gps is not None:
    shed_by_bus_gps = [(bus_idx, scenario.Pd_base[bus_idx], shed_vector_gps[i],
                        100*shed_vector_gps[i]/scenario.Pd_base[bus_idx] if scenario.Pd_base[bus_idx] > 0 else 0)
                       for i, bus_idx in enumerate(pq_bus_indices) if shed_vector_gps[i] > 1e-4]
    
    if shed_by_bus_gps:
        shed_by_bus_gps.sort(key=lambda x: x[2], reverse=True)
        print(f"\nGPS: Top buses by shedding amount:")
        print(f"{'Bus Idx':>8} {'Base Pd (MW)':>14} {'Shed (MW)':>12} {'% Shed':>10}")
        print("-" * 50)
        for bus_idx, base_pd, shed, pct in shed_by_bus_gps[:10]:
            print(f"{bus_idx:>8} {base_pd:>14.4f} {shed:>12.4f} {pct:>10.2f}%")
    else:
        print("GPS: No significant shedding (all < 0.0001 MW)")

## Summary: Current Interpretation

### What this notebook now represents

- It uses the corrected single-scenario extraction path.
- The optimizer now uses generator deviation cost, load shedding cost, and a wildfire-aware branch loading penalty.
- Thermal overload is no longer included as a separate objective term.

### What to look for when presenting results

- Compare the physical baseline wildfire cost from stored voltages against the surrogate wildfire cost from predicted voltages.
- Treat `validation_all_passed = False` as a model-quality signal, not a pipeline wiring failure.
- If optimization stops at 0 iterations, the surrogate is not providing a useful descent direction under the wildfire objective either.
- If GPS produces very large wildfire cost under the extended dispatch space, present that as instability.


In [ ]:
print("\n" + "=" * 70)
print("NOTEBOOK SUMMARY")
print("=" * 70)

print("\n[OK] Current notebook workflow:")
print("  1. Load a batched IEEE-30 test sample")
print("  2. Extract a single graph from the batch")
print("  3. Build PV-only and extended dispatch decision spaces")
print("  4. Load pretrained GNN and optional GPS checkpoints")
print("  5. Evaluate wildfire risk metrics")
print("  6. Run Phase 1 and Phase 2 optimization experiments")
print("  7. Compare GNN and GPS behavior under the wildfire objective")

print("\nKey presentation points:")
print(f"  Validation all_passed: {validation_all_passed}")
print(f"  Phase 1 objective improvement: {comparison['improvement']['objective_pct']:.2f}%")
print(f"  Phase 1 wildfire reduction: {comparison['improvement']['wildfire_pct']:.2f}%")
print(f"  Extended decision dimension: {decision_spec_extended.n_total}")
if has_gps:
    print(f"  GNN extended objective: {result_gnn_shedding['obj_opt']:.4f}")
    print(f"  GPS extended objective: {result_gps_shedding['obj_opt']:.4f}")

print("\nCaution:")
print("  Present this notebook as a wildfire-aware surrogate optimization prototype, not as a final AC-OPF-quality result.")
print("  If you want fresh figures for a presentation, rerun the notebook so the outputs match the current wildfire objective.")
print("\n" + "=" * 70)
